## Ingestão de dados - produtos

#### 1.Criar sessão spark e carregar configurações

In [ ]:
import sys
sys.path.append("/app/pipeline_spark/")

from utils import create_spark_session, load_config, save_table

spark = create_spark_session("produtos")
config = load_config()

db = config["sqlserver"]
tables = config["tables"]
jdbc_url = f"jdbc:sqlserver://{db['host']}:{db['port']};databaseName={db['database']}"

#### 2.Executar consulta SQL

In [ ]:
tabela_nome = "produtos"
query = ""

if tabela_nome in [table["name"] for table in tables]:

    query = f"""
        SELECT * FROM {tabela_nome}
    """

#### 3.Armazenar dados na camada bronze

In [ ]:
df = spark.read \
     .format("jdbc") \
     .option("url", jdbc_url) \
     .option("query", query) \
     .option("user", db["user"]) \
     .option("password", db["password"]) \
     .option("driver", db["driver"]) \
     .option("encrypt", "true") \
     .option("trustServerCertificate", "true") \
     .load()

table_config = next(
    table for table in tables
    if table["name"] == tabela_nome
)

save_table(
    df=df,
    table_config=table_config,
    config=config,
    spark=spark,
    layer="bronze"
)